In [1]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [26]:
event_log_name = "gigantic"
log_path = f"./.out/eventlogs/{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

parsing log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

In [27]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [28]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
# if True:
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

In [29]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


Loading conformance checking results from gigantic_5000_conformance_results.pkl
Conformance checking results loaded from gigantic_5000_conformance_results.pkl


In [30]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


/tmp/ipykernel_3978859/1810640346.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [31]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
# filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.2].sort_values(by='confidence', ascending=False)
print(f"Filtered Metrics DataFrame: {event_log_name}")
display(filtered_metrics_df)

Filtered Metrics DataFrame: gigantic


,support,confidence
"Responded Existence[Activity BJ, Activity B] | |",0.0754,0.994723
"Response[Activity BJ, Activity B] | |",0.0754,0.994723
"Response[Activity BN, Activity B] | |",0.0504,0.992126
"Responded Existence[Activity BN, Activity B] | |",0.0504,0.992126
"Responded Existence[Activity BN, Activity BJ] | |",0.0504,0.992126
...,...,...
"Chain Precedence[Activity C, Activity G] | |",0.0628,0.884507
"Precedence[Activity A, Activity BZ] | |",0.1240,0.844687
"Chain Response[Activity BZ, Activity B] | |",0.1236,0.841962
"Alternate Precedence[Activity A, Activity BZ] | |",0.1220,0.831063


In [32]:
raise KeyboardInterrupt("\nStopping execution after displaying filtered metrics DataFrame.\nChoose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.\nThen run the cells below again to see the results of the selected constraints.")

KeyboardInterrupt: 
Stopping execution after displaying filtered metrics DataFrame.
Choose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.
Then run the cells below again to see the results of the selected constraints.

# Constraints with low support and high confidence
1. Responded Existence[Activity BJ, Activity B] | |	0.0754	0.9947229551451188

In [ ]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    "Responded Existence[Activity BZ, Activity A] | |", # Gigantic
    # "Responded Existence[Activity Q, Activity O] | |", # wide
    ]

In [50]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Responded Existence[Activity BZ, Activity A] | |    685
dtype: int64
[6, 14, 26, 31, 35, 37, 41, 56, 58, 59, 68, 69, 77, 80, 82, 96, 104, 121, 133, 136, 138, 149, 156, 168, 177, 180, 183, 184, 192, 195, 198, 199, 203, 215, 232, 234, 241, 263, 265, 270, 280, 285, 291, 295, 306, 308, 311, 318, 328, 334, 339, 342, 349, 362, 365, 375, 382, 391, 397, 402, 430, 440, 442, 448, 454, 458, 463, 468, 475, 478, 484, 487, 499, 517, 533, 536, 553, 566, 567, 576, 577, 591, 593, 594, 602, 605, 607, 669, 676, 693, 697, 700, 701, 724, 732, 743, 749, 754, 761, 768, 772, 779, 790, 793, 802, 809, 812, 813, 822, 835, 848, 852, 855, 861, 875, 892, 893, 900, 903, 917, 923, 939, 953, 972, 978, 986, 996, 1004, 1005, 1011, 1030, 1035, 1038, 1040, 1045, 1050, 1062, 1067, 1070, 1076, 1081, 1089, 1095, 1105, 1109, 1150, 1162, 1164, 1169, 1176, 1179, 1181, 1184, 1198, 1209, 1211, 1219, 1237, 1254, 1259, 1261, 1268, 1270, 1282, 1295, 1299, 1302, 1303, 1309, 1311, 1317, 1329, 1349, 1355, 1360, 1383, 1388, 1390, 1396, 

In [51]:
print("END")

END


In [52]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)